# Notebook: TiDE v5

**What is different from TiDE v3:**

| | TiDE v3 | TiDE v5 |
|--|---------|----------|
| Data | Old (DDD inflated, outliers) | Clean (v3 data) |
| Promotion | Marginal | Included (0.5% proven gain) |
| Fourier | No | Added (0.7% gain, helps Month 6) |
| Epochs | 300 | 300 |
| Lookback | 18 months | 18 months |

**Target: WAPE < 8% (50% better than baseline)**

## Step 1 — Imports

In [2]:
import sys, warnings, json
sys.path.append("../03_scripts")
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from pathlib import Path
from evaluate import wape, macro_wape, full_scorecard
from utils import make_submission
from darts import TimeSeries
from darts.models import TiDEModel

OUTPUT = Path("../04_outputs/tide")
FINAL  = Path("../04_outputs/final")
FINAL.mkdir(exist_ok=True)
print("All imports OK")

All imports OK


## Step 2 — Load Clean Data

Using `master_train_v3.csv` — the cleaned version with DDD normalised, outliers capped, and smart lag filling.

In [3]:
master   = pd.read_csv("../01_input/processed/master_train_v3.csv", low_memory=False)
test     = pd.read_csv("../01_input/processed/master_test_v3.csv",  low_memory=False)
gne      = master[master["flag_competitor"]=="N"].copy()
gne_test = test.copy()
gne["date"]      = pd.to_datetime(gne["date_year_month"].astype(str),      format="%Y%m")
gne_test["date"] = pd.to_datetime(gne_test["date_year_month"].astype(str), format="%Y%m")
print(f"Train: {len(gne):,} rows | Test: {len(gne_test):,} rows")

Train: 29,200 rows | Test: 3,840 rows


## Step 3 — Load Updated Feature Selection

Features updated after re-running analysis on clean data:
- Promotion now positive (0.5% gain)
- Fourier seasonality added (0.7% gain)
- Price excluded (negative value)

In [4]:
with open("../05_documents/final_feature_selection.json") as f:
    sel = json.load(f)
FUTURE_COLS = sel["FINAL_FUTURE_COLS"]
PAST_COLS   = sel["FINAL_PAST_COLS"]

for c in FUTURE_COLS + PAST_COLS:
    if c not in gne.columns:      gne[c]      = 0
    gne[c] = gne[c].fillna(0)
for c in FUTURE_COLS:
    if c not in gne_test.columns: gne_test[c] = 0
    gne_test[c] = gne_test[c].fillna(0)

print(f"Future covariates: {len(FUTURE_COLS)}")
print(f"Past covariates  : {len(PAST_COLS)}")
print()
print("Key changes vs TiDE v3:")
print("  + Promotion included (rep_calls_adstock, digital_adstock, copay, marketing_spend)")
print("  + Fourier sin/cos terms (seasonal encoding for Month 6 gap)")

Future covariates: 18
Past covariates  : 14

Key changes vs TiDE v3:
  + Promotion included (rep_calls_adstock, digital_adstock, copay, marketing_spend)
  + Fourier sin/cos terms (seasonal encoding for Month 6 gap)


## Step 4 — Build TimeSeries Objects

In [5]:
train_series, fut_cov_train, past_cov_train, group_ids = [], [], [], []

for (bid, eid), grp in gne.groupby(["product_brand_id","ecosystem_id"]):
    grp = grp.sort_values("date").set_index("date")
    ts  = TimeSeries.from_series(grp["iqvia_sales_qty_eqv"].fillna(0), freq="MS")
    fc  = TimeSeries.from_dataframe(grp[[c for c in FUTURE_COLS if c in grp.columns]].fillna(0), freq="MS")
    pc  = TimeSeries.from_dataframe(grp[[c for c in PAST_COLS   if c in grp.columns]].fillna(0), freq="MS")
    train_series.append(ts)
    fut_cov_train.append(fc)
    past_cov_train.append(pc)
    group_ids.append((bid, eid))

print(f"Series: {len(train_series)} | Length: {min(len(t) for t in train_series)}-{max(len(t) for t in train_series)} months")

Series: 640 | Length: 34-48 months


In [6]:
# Extend future covariates through Jan-Jun 2025 horizon
fut_cov_full = []
for i, (bid, eid) in enumerate(group_ids):
    test_grp = gne_test[
        (gne_test["product_brand_id"]==bid) &
        (gne_test["ecosystem_id"]==eid)
    ].sort_values("date").set_index("date")
    if len(test_grp) == 0:
        fut_cov_full.append(fut_cov_train[i])
        continue
    fc_h = TimeSeries.from_dataframe(
        test_grp[[c for c in FUTURE_COLS if c in test_grp.columns]].fillna(0), freq="MS")
    fut_cov_full.append(fut_cov_train[i].append(fc_h))
print(f"Horizon covariates ready: {len(fut_cov_full)} series")

Horizon covariates ready: 640 series


## Step 5 — Train / Validation Split

In [7]:
VAL_MONTHS = 6
INPUT_LEN  = 18
MIN_LEN    = INPUT_LEN + VAL_MONTHS

train_split    = [ts[:-VAL_MONTHS] for ts in train_series]
val_split      = [ts[-VAL_MONTHS:]  for ts in train_series]
fc_train_split = [fc[:-VAL_MONTHS]  for fc in fut_cov_train]
pc_train_split = [pc[:-VAL_MONTHS]  for pc in past_cov_train]

valid_idx      = [i for i,ts in enumerate(train_split) if len(ts) >= MIN_LEN]
train_split    = [train_split[i]    for i in valid_idx]
val_split      = [val_split[i]      for i in valid_idx]
fc_train_split = [fc_train_split[i] for i in valid_idx]
pc_train_split = [pc_train_split[i] for i in valid_idx]
group_ids_v    = [group_ids[i]      for i in valid_idx]

print(f"Training series : {len(train_split)}")
print(f"Excluded (short): {640-len(train_split)}")
print(f"Min train length: {min(len(t) for t in train_split)} months")

Training series : 640
Excluded (short): 0
Min train length: 28 months


## Step 6 — Train TiDE v5

⏱ 300 epochs — expect 40-60 minutes.

In [8]:
model_v5 = TiDEModel(
    input_chunk_length=INPUT_LEN,
    output_chunk_length=6,
    num_encoder_layers=2,
    num_decoder_layers=2,
    decoder_output_dim=16,
    hidden_size=128,
    temporal_width_past=4,
    temporal_width_future=4,
    dropout=0.1,
    batch_size=64,
    n_epochs=300,
    add_encoders={
        "cyclic": {"future": ["month"]},
        "datetime_attribute": {"future": ["month", "quarter"]},
    },
    random_state=42,
    pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": True},
)

print("TiDE v5 — starting training...")
model_v5.fit(
    series=train_split,
    future_covariates=fc_train_split,
    past_covariates=pc_train_split,
    verbose=True,
)
print("Training complete!")
model_v5.save(str(OUTPUT / "tide_v5_model"))
print("Model saved.")

TiDE v5 — starting training...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                  ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ criterion             │ MSELoss          │      0 │ train │     0 │
│ 1  │ train_criterion       │ MSELoss          │      0 │ train │     0 │
│ 2  │ val_criterion         │ MSELoss          │      0 │ train │     0 │
│ 3  │ train_metrics         │ MetricCollection │      0 │ train │     0 │
│ 4  │ val_metrics           │ MetricCollection │      0 │ train │     0 │
│ 5  │ past_cov_projection   │ _ResidualBlock   │  2.5 K │ train │     0 │
│ 6  │ future_cov_projection │ _ResidualBlock   │  3.6 K │ train │     0 │
│ 7  │ encoders              │ Sequential       │  113 K │ train │     0 │
│ 8  │ decoders              │ Sequential       │ 90.8 K │ train │     0 │
│ 9  │ temporal_decoder      │ _ResidualBlock   │    726 │ train │     0 │
│ 10 │ lookback_skip         │ Linear           │    114 │ train │     0 │
└────┴───────────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 211 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 211 K                                                                                                
Total estimated model params size (MB): 1.693                                                                      
Modules in train mode: 57                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=300` reached.


Training complete!
Model saved.


## Step 7 — Validate & Compare All Versions

In [9]:
fc_for_val = [fut_cov_train[i] for i in valid_idx]
pc_for_val = [past_cov_train[i][:-VAL_MONTHS] for i in valid_idx]

val_preds = model_v5.predict(
    n=VAL_MONTHS, series=train_split,
    future_covariates=fc_for_val,
    past_covariates=pc_for_val,
)

y_true    = np.concatenate([ts.values().flatten() for ts in val_split])
y_pred    = np.concatenate([ts.values().flatten() for ts in val_preds])
brand_ids = np.concatenate([[gid[0]]*VAL_MONTHS for gid in group_ids_v])
steps     = np.tile(range(1, VAL_MONTHS+1), len(group_ids_v))

results, brand_wapes, step_wapes = full_scorecard(y_true, y_pred, brand_ids, None, steps)

with open(OUTPUT / "validation_scores_v3.json") as f: v3 = json.load(f)

print(f"\n=== ALL TiDE VERSIONS ===")
print(f"TM1 Baseline : 0.1370  (13.7%)")
print(f"TiDE v1      : 0.1106  (11.1%)")
print(f"TiDE v2      : 0.1018  (10.2%)")
print(f"TiDE v3      : {v3['WAPE (overall)']:.4f}  ({v3['WAPE (overall)']*100:.1f}%)  old data")
print(f"TiDE v5      : {results['WAPE (overall)']:.4f}  ({results['WAPE (overall)']*100:.1f}%)  clean data  <-- NEW")

imp = (v3["WAPE (overall)"] - results["WAPE (overall)"]) / v3["WAPE (overall)"] * 100
print(f"\nImprovement over v3: {imp:.1f}%")
print(f"vs TM1 baseline    : {(0.137-results['WAPE (overall)'])/0.137*100:.1f}% better")

print("\nHorizon step breakdown:")
print(step_wapes.to_string())

with open(OUTPUT / "validation_scores_v5.json", "w") as f:
    json.dump({k: round(float(v),5) for k,v in results.items()}, f, indent=2)
print("\nScores saved.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

=== Overall Scores ===
  WAPE (overall): 0.0356
  MACRO-WAPE (by brand): 0.0465
  sMAPE: 0.0490
  Bias (avg): -2.9791

=== WAPE by Brand ===
brand
5101    0.056558
5102    0.064057
5105    0.009306
5109    0.008024
5112    0.006606
5116    0.054960
5117    0.091846
5118    0.080464

=== WAPE by Horizon Step (month 1..6) ===
step
1    0.036767
2    0.034079
3    0.036540
4    0.037156
5    0.033703
6    0.035660

=== ALL TiDE VERSIONS ===
TM1 Baseline : 0.1370  (13.7%)
TiDE v1      : 0.1106  (11.1%)
TiDE v2      : 0.1018  (10.2%)
TiDE v3      : 0.0818  (8.2%)  old data
TiDE v5      : 0.0356  (3.6%)  clean data  <-- NEW

Improvement over v3: 56.4%
vs TM1 baseline    : 74.0% better

Horizon step breakdown:
step
1    0.036767
2    0.034079
3    0.036540
4    0.037156
5    0.033703
6    0.035660

Scores saved.


## Step 8 — Generate Final Predictions & Apply MinTrace

In [10]:
final_preds = model_v5.predict(
    n=6, series=train_series,
    future_covariates=fut_cov_full,
    past_covariates=past_cov_train,
)
predictions = np.concatenate([ts.values().flatten() for ts in final_preds])
predictions = np.clip(predictions, 0, None)

sub = pd.read_csv("../01_input/raw/sample_submission.csv")
make_submission(sub["row_id"].values, predictions, OUTPUT / "tide_v5_submission.csv")
print("TiDE v5 raw predictions saved.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

Submission saved: ../04_outputs/tide/tide_v5_submission.csv  (3840 rows, min=0.00)
TiDE v5 raw predictions saved.


In [11]:
# Apply MinTrace directly
preds_df  = pd.read_csv(OUTPUT / "tide_v5_submission.csv")
test_meta = pd.read_csv("../01_input/raw/test_features.csv")
prod      = pd.read_csv("../01_input/raw/dim_product.csv")[["product_brand_id","product_brand_name"]]

preds_df = preds_df.merge(test_meta[["row_id","date_year_month","product_brand_id","ecosystem_id"]], on="row_id", how="left")
preds_df = preds_df.merge(prod, on="product_brand_id", how="left")

reconciled = []
for month in sorted(preds_df["date_year_month"].unique()):
    for brand_id in preds_df["product_brand_id"].unique():
        sub_m = preds_df[(preds_df["date_year_month"]==month) & (preds_df["product_brand_id"]==brand_id)].copy()
        if len(sub_m) == 0: continue
        y_hat = sub_m["forecast_units_eqv"].values.copy()
        n = len(y_hat)
        S = np.vstack([np.ones((1,n)), np.eye(n)])
        y_all = np.concatenate([[y_hat.sum()], y_hat])
        P = np.linalg.pinv(S.T @ S) @ S.T
        sub_m["forecast_reconciled"] = (S @ P @ y_all)[1:].clip(min=0)
        reconciled.append(sub_m)

final = pd.concat(reconciled, ignore_index=True)

make_submission(final["row_id"].values, final["forecast_reconciled"].values,
                FINAL / "tide_v5_mintrace_submission.csv")
make_submission(final["row_id"].values, final["forecast_reconciled"].values,
                FINAL / "final_submission.csv")

print(f"\n=== FINAL RESULT ===")
print(f"TiDE v5 WAPE        : {results['WAPE (overall)']*100:.2f}%")
print(f"vs TM1 baseline     : {(0.137-results['WAPE (overall)'])/0.137*100:.1f}% better")
print(f"Submission saved    : 04_outputs/final/final_submission.csv")

Submission saved: ../04_outputs/final/tide_v5_mintrace_submission.csv  (3840 rows, min=0.00)
Submission saved: ../04_outputs/final/final_submission.csv  (3840 rows, min=0.00)

=== FINAL RESULT ===
TiDE v5 WAPE        : 3.56%
vs TM1 baseline     : 74.0% better
Submission saved    : 04_outputs/final/final_submission.csv
